# Σύστημα Β – Βήμα 1: προεπεξεργασία και πρώτη εικόνα του προβλήματος
Δεύτερο σύστημα: σύνολο δεδομένων **cybersecurity_intrusion_data**. Κάθε γραμμή είναι μια συνεδρία (session) χρήστη και ο στόχος είναι ένας:
`attack_detected` (1 = επίθεση, 0 = κανονική).

1. Φόρτωση του έτοιμου αρχείου και προσθήκη του `browser_type`.
2. Διαχωρισμός σε train / validation / test.
3. Μετασχηματισμός των εισόδων σε μορφή για το νευρωνικό δίκτυο.
4. Γιατί 9 είσοδοι και όχι 8: ο έλεγχος του `browser_type`.
5. Πόσο εύκολο είναι το πρόβλημα; (απλοί κανόνες και κλασικά μοντέλα, **μόνο στο validation**)
6. Αποθήκευση.

Το test set **δεν το αγγίζουμε** εδώ· θα χρησιμοποιηθεί μία φορά, στην τελική αξιολόγηση. Χρόνος: κάτω από 1 λεπτό.

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score

DATA, MODELS = "../data/", "../models/"
SEED = 42

TARGET = "attack_detected"
NUM_COLS = ["network_packet_size", "login_attempts", "session_duration", "ip_reputation_score", "failed_logins"]
BIN_COLS = ["unusual_time_access"]                                       # ήδη 0/1
CAT_COLS = ["protocol_type", "encryption_used", "browser_type"]          # κείμενο
FEATURES = NUM_COLS + BIN_COLS + CAT_COLS

## Β1.1 Φόρτωση
Ο καθαρισμός έγινε ήδη στο notebook `01_prepare_data` (αρχείο `intrusion_clean.csv`: 8 είσοδοι + στόχος) και η διερεύνηση στο `02_eda` (figures 01, 03, 05, 06). Εδώ **δεν τα ξανακάνουμε**.

Το `intrusion_clean.csv` δεν περιέχει το `browser_type`, που είχε μείνει εκτός στην αρχική επιλογή στηλών. Το παίρνουμε από το αρχικό αρχείο, αφού πρώτα επιβεβαιώσουμε ότι οι γραμμές είναι οι ίδιες και με την ίδια σειρά. Το `session_id` είναι απλός κωδικός συνεδρίας και δεν χρησιμοποιείται.

`keep_default_na=False`: η τιμή `"None"` στο `encryption_used` σημαίνει «χωρίς κρυπτογράφηση» και πρέπει να μείνει κείμενο.

In [ ]:
df = pd.read_csv(DATA + "intrusion_clean.csv", keep_default_na=False)
raw = pd.read_csv(DATA + "cybersecurity_intrusion_data.csv")
same_rows = len(raw) == len(df) and np.allclose(raw["session_duration"], df["session_duration"]) \
            and (raw[TARGET].to_numpy() == df[TARGET].to_numpy()).all()
assert same_rows, "Το αρχικό αρχείο δεν έχει τις ίδιες γραμμές με το intrusion_clean.csv – ξανατρέξε το 01_prepare_data"
assert raw["browser_type"].notna().all(), "Κενές τιμές στο browser_type"
df["browser_type"] = raw["browser_type"].to_numpy()

missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
assert not missing, f"Λείπουν στήλες: {missing}"
assert df[NUM_COLS + BIN_COLS + [TARGET]].notna().all().all(), "Υπάρχουν κενές τιμές – ξανατρέξε το 01_prepare_data"
print("Σχήμα:", df.shape, "(αναμενόμενο: 9537 γραμμές, 10 στήλες)")
print(f"Ποσοστό επιθέσεων: {df[TARGET].mean():.1%}")
print("Διπλότυπες γραμμές:", int(df.duplicated().sum()))
print("browser_type:", df["browser_type"].value_counts().to_dict())

## Β1.2 Διαχωρισμός train / validation / test (70% / 15% / 15%)
Εδώ δεν υπάρχει έτοιμο test set όπως στο UNSW-NB15, οπότε το φτιάχνουμε εμείς. Με `stratify`, ώστε το ποσοστό επιθέσεων να είναι ίδιο και στα τρία κομμάτια.

In [ ]:
train_df, rest_df = train_test_split(df, test_size=0.30, stratify=df[TARGET], random_state=SEED)
val_df, test_df = train_test_split(rest_df, test_size=0.50, stratify=rest_df[TARGET], random_state=SEED)
frames = {"train": train_df, "val": val_df, "test": test_df}
for s, f in frames.items():
    print(f"{s:5s}: {len(f):5d} γραμμές | επιθέσεις {f[TARGET].mean():.1%}")

## Β1.3 Μετασχηματισμός εισόδων
Ίδιοι κανόνες με το Σύστημα Α, και όλα «μαθαίνονται» **μόνο από το train**:
- αριθμητικές με πολύ ασύμμετρη κατανομή (skewness > 2) → `log(1+x)` και μετά κλιμάκωση·
- υπόλοιπες αριθμητικές → κλιμάκωση (μέσος 0, τυπική απόκλιση 1)·
- δυαδικές (0/1) → όπως είναι·
- κατηγορικές → one-hot.

In [ ]:
skew = train_df[NUM_COLS].skew()
LOG_COLS = skew[skew > 2].index.tolist()
PLAIN_COLS = [c for c in NUM_COLS if c not in LOG_COLS]
print("Ασυμμετρία:", skew.round(2).to_dict())
print("Με log:", LOG_COLS)

steps = []
if LOG_COLS:
    steps.append(("log", Pipeline([("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                                   ("scale", StandardScaler())]), LOG_COLS))
if PLAIN_COLS:
    steps.append(("num", StandardScaler(), PLAIN_COLS))
steps.append(("bin", "passthrough", BIN_COLS))
steps.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS))
preprocessor = ColumnTransformer(steps)

X = {"train": preprocessor.fit_transform(train_df).astype("float32")}            # fit ΜΟΝΟ στο train
X["val"] = preprocessor.transform(val_df).astype("float32")
X["test"] = preprocessor.transform(test_df).astype("float32")
y = {s: f[TARGET].to_numpy("int64") for s, f in frames.items()}
feature_names = preprocessor.get_feature_names_out().tolist()
short = [f.split("__")[-1] for f in feature_names]
print("\nΣχήματα:", {s: X[s].shape for s in X})
print("Είσοδοι δικτύου:", short)

## Β1.4 Γιατί 9 είσοδοι και όχι 8: ο έλεγχος του `browser_type`
Η αρχική επιλογή είχε 8 εισόδους. Ελέγχουμε στο validation αν το `browser_type` προσθέτει πληροφορία: ίδιο μοντέλο (Gradient Boosting), ίδιες γραμμές, με και χωρίς τη στήλη.

In [ ]:
def metrics(y_true, p):
    pred = p >= 0.5
    return {"accuracy": accuracy_score(y_true, pred), "ROC-AUC": roc_auc_score(y_true, p),
            "recall (επιθέσεις)": recall_score(y_true, pred),
            "FPR (ψευδείς συναγερμοί)": ((pred == 1) & (y_true == 0)).sum() / (y_true == 0).sum()}

without = [i for i, f in enumerate(short) if not f.startswith("browser_type")]
check = {}
for name, cols in [("8 είσοδοι (χωρίς browser_type)", without), ("9 είσοδοι (με browser_type)", list(range(len(short))))]:
    gb = GradientBoostingClassifier(random_state=SEED).fit(X["train"][:, cols], y["train"])
    check[name] = metrics(y["val"], gb.predict_proba(X["val"][:, cols])[:, 1])
check = pd.DataFrame(check).T.round(4)
check.to_csv(MODELS + "intrusion_browser_type_check.csv")
print("Gradient Boosting στο validation:"); print(check.to_string())

both = pd.concat([train_df, val_df])
rate = pd.DataFrame({"πλήθος": both.groupby("browser_type").size(), "ποσοστό επιθέσεων": both.groupby("browser_type")[TARGET].mean()})
print("\nΠοσοστό επιθέσεων ανά browser_type (train + validation):"); print(rate.round(3).to_string())

## Β1.5 Πόσο εύκολο είναι το πρόβλημα;
Πριν φτιάξουμε νευρωνικό, μετράμε τι πετυχαίνουν πιο απλά πράγματα. Αν ένας κανόνας με λίγες ερωτήσεις έφτανε κοντά στα καλύτερα μοντέλα, το δίκτυο δεν θα πρόσθετε κάτι
(το ερώτημα που έθεσε ο επιβλέπων για το UNSW-NB15).

Όλα μετρώνται στο **validation**, με τις 9 εισόδους.

In [ ]:
def scores(model):
    return metrics(y["val"], model.predict_proba(X["val"])[:, 1])

rows = {"Πάντα «κανονική»": {"accuracy": (y["val"] == 0).mean(), "ROC-AUC": 0.5, "recall (επιθέσεις)": 0.0, "FPR (ψευδείς συναγερμοί)": 0.0}}

single = {}
for i, f in enumerate(short):
    m = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(X["train"][:, [i]], y["train"])
    single[f] = accuracy_score(y["val"], m.predict(X["val"][:, [i]]))
best = max(single, key=single.get)
m = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(X["train"][:, [short.index(best)]], y["train"])
rows[f"Καλύτερος κανόνας με ένα χαρακτηριστικό ({best})"] = metrics(y["val"], m.predict_proba(X["val"][:, [short.index(best)]])[:, 1])

trees = {}
for depth in [2, 3, 4, 5]:
    trees[depth] = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(X["train"], y["train"])
    rows[f"Δέντρο απόφασης βάθους {depth}"] = scores(trees[depth])
rows["Logistic Regression"] = scores(LogisticRegression(max_iter=1000).fit(X["train"], y["train"]))
rows["Random Forest (300 δέντρα)"] = scores(RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED).fit(X["train"], y["train"]))
rows["Gradient Boosting"] = scores(GradientBoostingClassifier(random_state=SEED).fit(X["train"], y["train"]))

simple = pd.DataFrame(rows).T.round(4)
simple.to_csv(MODELS + "intrusion_simple_models_validation.csv")
print(simple.to_string())
print("\nAccuracy κάθε χαρακτηριστικού μόνο του (validation):")
print(pd.Series(single).sort_values(ascending=False).round(4).to_string())

### Οι κανόνες των δέντρων
Για να δούμε **ποιες** ερωτήσεις κάνουν. Οι αριθμοί στα όρια είναι στην κλίμακα μετά τον μετασχηματισμό (όχι στις αρχικές μονάδες).

In [ ]:
for depth in [3, 4]:
    print(f"--- Δέντρο βάθους {depth} ---")
    print(export_text(trees[depth], feature_names=short, decimals=2))

## Β1.6 Αποθήκευση
Τα έτοιμα δεδομένα, ο preprocessor (για νέες συνεδρίες) και οι ρυθμίσεις. Αποθηκεύονται και οι αριθμοί γραμμών κάθε κομματιού, ώστε τα επόμενα notebooks να βρίσκουν τις ίδιες γραμμές.

In [ ]:
np.savez_compressed(DATA + "intrusion_prepared.npz",
                    X_train=X["train"], X_val=X["val"], X_test=X["test"],
                    y_train=y["train"], y_val=y["val"], y_test=y["test"],
                    idx_train=train_df.index.to_numpy(), idx_val=val_df.index.to_numpy(), idx_test=test_df.index.to_numpy())
joblib.dump(preprocessor, MODELS + "intrusion_preprocessor.joblib")
meta = {"target": TARGET, "num_cols": NUM_COLS, "bin_cols": BIN_COLS, "cat_cols": CAT_COLS, "log_cols": LOG_COLS,
        "features": FEATURES, "feature_names": feature_names, "random_state": SEED,
        "split": {"train": 0.70, "val": 0.15, "test": 0.15},
        "sizes": {s: int(len(f)) for s, f in frames.items()}}
with open(MODELS + "intrusion_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Αποθηκεύτηκαν: data/intrusion_prepared.npz, models/intrusion_preprocessor.joblib, models/intrusion_meta.json")